# Curricula: preprocessing

First half of the CV build, everything that does not need the KB mapping: normalize the anonymized
CVs, drop duplicates and CVs without skills, assign `id_c`, and write the input of the mapping
notebooks.

- Load `anonymized_cvs.csv`, normalize the experience buckets and job categories, flatten the
  nested `anonymized_cv` field, merge hard, soft and other skills into one `skills` set, drop the
  fields that are not exported (demographics included) and attach `edu_eqf` from
  `mapping/education_map.csv`. The source row id is used only for that join; after it, `id_c` is
  the only candidate id.
- Clean the language names with `language_map`; unmapped values are dropped.
- Drop exact duplicates, then CVs with no skill, printing the counts. `id_c` is then assigned as
  0..N-1.

Writes `mapping/preprocessed_cvs.json` for the mapping notebooks and `mapping/stage1_cvs.json`
for `3_build_cvs.ipynb`.

Next: `1_translate_skills.ipynb`, `2_skill_extract.ipynb`, `3_build_cvs.ipynb`.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from ast import literal_eval

In [ ]:
# Normalize raw professional-experience buckets into compact, consistent ranges.
experience_map = {
    "4 years or less of experience": "0 - 4 years",
    "15+ years of experience": "+15 years",
    "5-9 years of experience": "5 - 9 years",
    "10-14 years of experience": "10 - 14 years"
}

# Map the job-category labels to ESCO-style occupational categories.
job_category = {
    "Food Processing, woodworking, garment and other Craft": "Food processing, wood working, garment and other craft and related trades workers",
    "Business and administration": "Business and administration professionals",
    "Clerical support": "General and keyboard clerks",
    "Sales": "Sales workers",
    "ICT": "Information and communications technicians",
    "Science and engineering": "Science and engineering professionals",
    "Construction, manufacturing and transport": "Labourers in mining, construction, manufacturing and transport",
    "Personal service": "Personal service workers",
    "Food preparation": "Food preparation assistants",
    "Health": "Health professionals",
    "Teaching": "Teaching professionals",
    "Hospitality, retail and other services": "Hospitality, retail and other services managers",
    "Legal, social and cultural": "Legal, social and cultural professionals",
    "Production and specialized services": "Production and specialised services managers",
    "Agricultural, forestry and fishery": "Agricultural, forestry and fishery labourers",
    "Public officials": "Protective services workers",
    "Handicraft and printing": "Handicraft and printing workers",
    "Cleaning": "Cleaners and helpers",
    "Armed forces": "Commissioned armed forces officers",
    "Personal care": "Personal care workers",
    "Protective services": "Protective services workers"
}

# Clean language labels found in CVs; values set to None are discarded as non-language noise.
language_map = {
    "Castilian": "Spanish",
    "French Medium-high": "French",
    "cambridge": "English",
    "Spanisih": "Spanish",
    "Valencian": "Spanish",
    "Arab": "Arabic",
    "Mandarin": "Chinese",
    "Swiss German": "German",
    "Englisih": "English",
    "Darija": "Arabic",
    "Native Spanish.": "Spanish",
    "English - Intermediate B2.": "English",
    "Italian - Intermediate B2.": "Italian",
    "Slovene": "Slovenian",
    "Cantonese": "Chinese",
    "German mother tongue": "German",
    "French C2 spoken and written": "French",
    "Spanish B1 spoken and written ÿ": "Spanish",
    "English C2 spoken and written": "English",
    "German: Customer Service (209H)": "German",
    "Spanish: Bilingual": "Spanish",
    "Basic-level English": "English",
    "Swidish": "Swedish",
    "Malaysian": "Malay",
    "Ukranian": "Ukrainian",
}


In [ ]:
edu_map = pd.read_csv("mapping/education_map.csv", encoding="utf-8", index_col=0)

cvs_dt = pd.read_csv(
    "../source/anonymized_cvs.csv",
    encoding="utf-8", converters={"job_category":lambda x: job_category[x],
                                  "years_professional_experience": lambda  x: experience_map[x]}).fillna("Not specified")


# Flatten the nested CV JSON, drop the fields that are not exported and attach the education
# level. The source row id is only used to join education_map.
cvs_dt = (
    pd
    .concat([
        cvs_dt,
        pd.json_normalize(cvs_dt["anonymized_cv"].apply(literal_eval))
    ], axis=1)
    .assign(skills=lambda df: (df["skills.hard"] + df["skills.soft"] + df["skills.others"]).apply(lambda x: tuple(sorted(set(x)))))
    .drop(columns=["anonymized_cv", "education_background", "sex_at_birth", "sexual_orientation", "age", "gender", "ethnic_group",
                   "religion", "disability", "agg_sexual_orientation","perceived_foreign", "agg_age", "agg_minority",
                   "professional_experience", "agg_perceived_foreign", "skills.hard", "skills.soft", "skills.others",])
    .merge(edu_map, how="left", left_index=True, right_on="index")
    .reset_index(drop=True)
)

In [ ]:
cvs_dt.head(10)

In [ ]:
# education_map must cover exactly the CVs of anonymized_cvs.
assert set(edu_map.index) == set(cvs_dt.index), f"education_map id_c mismatch vs anonymized_cvs: {set(edu_map.index) ^ set(cvs_dt.index)}"

In [ ]:
cvs_dt["languages"] = cvs_dt["skills.languages"].apply(
    lambda x: tuple(sorted({r for i in x if (r := language_map.get(i, i)) is not None}))
)
cvs_dt = cvs_dt.drop(columns=["skills.languages"])

## Drop duplicates

In [ ]:
print("Total rows", len(cvs_dt))
index_duplicated = cvs_dt[cvs_dt.duplicated()].index
cvs_dt = cvs_dt.drop(index=index_duplicated)
print("Duplicate cvs : ", len(index_duplicated))
print("Remaining rows", len(cvs_dt))

## Drop CVs with no skills

In [ ]:
print("Total rows", len(cvs_dt))
print("Candidate with 0 skills", sum(cvs_dt.skills.str.len() == 0))
cvs_dt = cvs_dt[cvs_dt.skills.str.len() > 0]
print("Remaining cvs", len(cvs_dt))

In [ ]:
cvs_dt["id_c"] = [*range(len(cvs_dt))]

# Two outputs of the same frame: preprocessed_cvs.json and stage1_cvs.json.

cvs_dt[["id_c", "job_category", "skills"]].to_json(
    "mapping/preprocessed_cvs.json", orient="records", indent=2)
cvs_dt.to_json("mapping/stage1_cvs.json", orient="records", indent=2)
print(f"{len(cvs_dt)} cvs -> mapping/preprocessed_cvs.json + mapping/stage1_cvs.json")